In [1]:
import sys
sys.path.append("../../")

%load_ext autoreload
%autoreload 2
    
import pandas as pd
import numpy as np
import optuna
import pickle
import matplotlib.pyplot as plt

from functools import partial

from simulator.simulation.modules import Campaign
from simulator.simulation.utils_visualization import data_prep_vis, plot_history_article
from simulator.simulation.simulate import simulate_campaign
from simulator.validation.check_results import autobidder_check

# import baselines:
from simulator.model.linear_bidder import LinearBidder
from simulator.model.ta_pid import TAPIDBidder
from simulator.model.m_pid import MPIDBidder
from simulator.model.mystique import Mystique
from simulator.model.broi_bidder import BROI
from simulator.model.traffic import Traffic
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', None)

In [2]:
auction_mode = 'FPA' # you may also choose the 'VCG' mode

# data paths
campaigns_path = f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_train_final.csv"
stats_path = f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_train_final.csv"

In [3]:
campaign_df = pd.read_csv(campaigns_path)
stats_df = pd.read_csv(stats_path)

In [4]:
from utils import DATA_DIR

In [5]:
# campaigns_path_test = f"../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_test_final.csv"
# stats_path_test = f"../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_test_final.csv"

campaigns_path_test = str(DATA_DIR / f"{auction_mode.lower()}" / f"campaigns_{auction_mode.lower()}_filtered_test_final.csv")
stats_path_test = str(DATA_DIR / f"{auction_mode.lower()}" / f"stats_{auction_mode.lower()}_filtered_test_final.csv")

In [6]:
from simulator.model.rlb_dp_bidder import RLBDPBidder

In [11]:
final_model_path = "best_params/rlb_dp_model_new_tuned.pkl"

In [12]:
res = autobidder_check(
    bidder=RLBDPBidder,
    params = {
        "input_campaigns": campaigns_path_test,
        "input_stats": stats_path_test,
        "model_path": final_model_path,
    },
    auction_mode=auction_mode,
)

In [13]:
hist_data = res['all_hist_data']

In [20]:
hist_data[1]['clicks_history'].sum()

np.float64(83.14148776462926)

In [22]:
hist_data[1]['clicks'].max()

np.float64(83.1414877646293)

In [18]:
data = (
        hist_data[5]
        .groupby("campaign_id")
        .agg(
            start_time=("campaign_start_time", "first"),
            end_time=("campaign_end_time", "first"),
            region_id=("region_id", "first"),
            clicks=('clicks', 'sum'),
            initial_balance=('initial_balance', 'first'),
            desired_clicks=('desired_clicks', 'first'),
            spend_history=('spend_history', lambda x: list(x)),
            clicks_history=('clicks_history', lambda x: list(x))
        )
    )

In [19]:
data

,start_time,end_time,region_id,clicks,initial_balance,desired_clicks,spend_history,clicks_history
campaign_id,,,,,,,,
38099586,529754472,529840872,652560,177.655984,552.96,110.0,"[1.048644535627098, 83.39086394054698, 24.1842...","[0.640115097644944, 1.316894065054288, 0.36084..."
